## Reproducing result figures and evaluation metrics

This notebook can be used to evaluate the different models by running new simulations and computing metrics and associated figures:
* State-averaged cumulative error $E(t)$
* Wasserstein distance $W_1$

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import tqdm
import h5py
import matplotlib.pyplot as plt

plt.rcParams.update({
  'mathtext.fontset': 'cm'
})

import numpy as np
import scipy
import jax
import jax.numpy as jnp
import jax.random as jnr

from flax import nnx
import orbax.checkpoint as ocp

jax.config.update(
  'jax_enable_x64', True
)

import models.time_solver as stepper
from models.l96 import (
    L96, 
    dynamical_solver,
    dynamical_solver_single
)
from models.mlp import (
    FwdMLP,
    ResMLP
)

In [ ]:
def run_system(
    eq: L96,
    solver,
    x_k: jnp.ndarray,
    y_j: jnp.ndarray,
    dt: float,
    t0: float,
    T: float,
    n_logs: int
):
    time = t0
    iters = int(T / dt)
    logs_freq = int(iters / n_logs)
    
    time_t = []
    x_k_t = []
    
    pbar = tqdm.tqdm(range(iters), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
    for i in pbar:
        x_k, y_j = solver(x_k, y_j, time)
        time += dt
        if i % logs_freq == 0:
            time_t.append(time)
    
            x_k_t.append(
                x_k
            )
    
    return (
        np.asarray(time_t),
        np.asarray(x_k_t),
    )

### Running evaluation with the two-timescales solver:

In [ ]:
decorr_t = 6.0
eval_n_decorr = 100
n_logs = 30000

cfg_name = 'l96'
data_path = os.path.join(os.path.join(os.getcwd(), '../data'), cfg_name)

with h5py.File(os.path.join(data_path, 'datasets.h5'), 'r') as f:
    dt = f.attrs['dt']
    n_steps = f.attrs['n_steps']
    n_trajs = f.attrs['n_trajs']
    source_val = f.attrs['source_val']
eq, time, x_k_0, y_j = L96.load(
    os.path.join(data_path, 'snapshot.h5')
)

def source_term(x_k, t):
    return source_val

solver = jax.jit(dynamical_solver(
    eq,
    stepper.RK4(dt),
    source_term
))

print('Running two-timescales system')
time_t, x_k_t = run_system(
    eq,
    solver,
    x_k=x_k_0,
    y_j=y_j,
    dt=dt,
    t0=time,
    T=decorr_t * eval_n_decorr,
    n_logs=n_logs
)

### Running evaluation with the single-timescale models (without and with SGS models):

In [ ]:
# "Single-timescale" model, with N_j = 0
eq_single = L96(
    b=eq.b,
    c=eq.c,
    h=eq.h,
    n_k=eq.n_k,
    n_j=0
)

dt_single = eq.n_j * dt

def source_nop(x_k, t):
    return source_term(x_k, t)

solver_nop = jax.jit(dynamical_solver_single(
    eq_single,
    stepper.RK4(dt_single),
    source_nop
))

print('Running single-timescale system') 
_, x_k_t_nop = run_system(
    eq_single,
    solver_nop,
    x_k=x_k_0,
    y_j=0,
    dt=dt_single,
    t0=time,
    T=decorr_t * eval_n_decorr,
    n_logs=n_logs
)

abstract_model = nnx.eval_shape(lambda: FwdMLP(
    in_features=eq.n_k,
    latent=16,
    out_features=eq.n_k,
    n_blocks=3,
    means=jnp.zeros((8,)), 
    stds=jnp.zeros((8,)),
    activation=nnx.relu,
    rngs=nnx.Rngs(42)
))

graph, abstract_state = nnx.split(abstract_model)

checkpoint_path = os.path.join(data_path, 'ref_checkpoint/')
checkpointer = ocp.Checkpointer(ocp.StandardCheckpointHandler())
import warnings
with warnings.catch_warnings(action='ignore'):
    state = checkpointer.restore(checkpoint_path, abstract_state)
eq_ref = nnx.merge(graph, state)

def source_ref(x_k, t):
    return source_term(x_k, t) + eq_ref(x_k)

solver_ref = jax.jit(dynamical_solver_single(
    eq_single,
    stepper.RK4(dt_single),
    source_ref
))

print('Running solver-based (ref) model')
_, x_k_t_ref = run_system(
    eq_single,
    solver_ref,
    x_k=x_k_0,
    y_j=0,
    dt=dt_single,
    t0=time,
    T=decorr_t * eval_n_decorr,
    n_logs=n_logs
)

graph, abstract_state = nnx.split(abstract_model)

checkpoint_path = os.path.join(data_path, 'state_checkpoint/')
checkpointer = ocp.Checkpointer(ocp.StandardCheckpointHandler())
import warnings
with warnings.catch_warnings(action='ignore'):
    state = checkpointer.restore(checkpoint_path, abstract_state)
eq_state = nnx.merge(graph, state)

def source_state(x_k, t):
    return source_term(x_k, t) + eq_state(x_k)

solver_state = jax.jit(dynamical_solver_single(
    eq_single,
    stepper.RK4(dt_single),
    source_state
))

print('Running emulator-based model (state loss)')
_, x_k_t_state = run_system(
    eq_single,
    solver_state,
    x_k=x_k_0,
    y_j=0,
    dt=dt_single,
    t0=time,
    T=decorr_t * eval_n_decorr,
    n_logs=n_logs
)

graph, abstract_state = nnx.split(abstract_model)

checkpoint_path = os.path.join(data_path, 'subgrid_checkpoint/')
checkpointer = ocp.Checkpointer(ocp.StandardCheckpointHandler())
import warnings
with warnings.catch_warnings(action='ignore'):
    state = checkpointer.restore(checkpoint_path, abstract_state)
eq_subgrid = nnx.merge(graph, state)

def source_subgrid(x_k, t):
    return source_term(x_k, t) + eq_subgrid(x_k)

solver_subgrid = jax.jit(dynamical_solver_single(
    eq_single,
    stepper.RK4(dt_single),
    source_subgrid
))

print('Running emulator-based model (subgrid loss)')
_, x_k_t_subgrid = run_system(
    eq_single,
    solver_subgrid,
    x_k=x_k_0,
    y_j=0,
    dt=dt_single,
    t0=time,
    T=decorr_t * eval_n_decorr,
    n_logs=n_logs
)

### Generating the metrics figure: $E(t)$ and $W_1$:

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(7.0, 3.5), dpi=120)

time_cdf = np.expand_dims(np.linspace(1, 1 + dt_single * x_k_t.shape[0], x_k_t.shape[0]), axis=1)

terr_nop     = np.cumsum(np.sqrt((x_k_t - x_k_t_nop    )**2), axis=0) / time_cdf
terr_ref     = np.cumsum(np.sqrt((x_k_t - x_k_t_ref    )**2), axis=0) / time_cdf
terr_state   = np.cumsum(np.sqrt((x_k_t - x_k_t_state  )**2), axis=0) / time_cdf
terr_subgrid = np.cumsum(np.sqrt((x_k_t - x_k_t_subgrid)**2), axis=0) / time_cdf

axs[0].plot(time_t, np.mean(terr_nop, axis=1), label=r'$\tau \equiv 0$', color='#999999', linestyle='--')
axs[0].plot(time_t, np.mean(terr_state, axis=1), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state}$', color='#E69F00')
axs[0].plot(time_t, np.mean(terr_subgrid, axis=1), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{sugrid}$', color='#D55E00')
axs[0].plot(time_t, np.mean(terr_ref, axis=1), label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{ref})$', color='#56B4E9')

axs[0].set_ylabel(r'$E(t)$', fontsize=15)
axs[0].set_xlabel(r'$t$', fontsize=15)
axs[0].tick_params(reset=True, axis='both', which='both', direction='in')
axs[0].legend(fontsize=12, frameon=False)

derr_nop     = [scipy.stats.wasserstein_distance(x_k_t[:, k], x_k_t_nop    [:, k]) for k in range(eq.n_k)]
derr_ref     = [scipy.stats.wasserstein_distance(x_k_t[:, k], x_k_t_ref    [:, k]) for k in range(eq.n_k)]
derr_state   = [scipy.stats.wasserstein_distance(x_k_t[:, k], x_k_t_state  [:, k]) for k in range(eq.n_k)]
derr_subgrid = [scipy.stats.wasserstein_distance(x_k_t[:, k], x_k_t_subgrid[:, k]) for k in range(eq.n_k)]

axs[1].plot(np.arange(eq.n_k) + 1, derr_nop, label=r'$\tau \equiv 0$', color='#999999', linestyle='--')
axs[1].plot(np.arange(eq.n_k) + 1, derr_state, label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state}$', color='#E69F00')
axs[1].plot(np.arange(eq.n_k) + 1, derr_subgrid, label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{sugrid}$', color='#D55E00')
axs[1].plot(np.arange(eq.n_k) + 1, derr_ref, label=r'$\tau \equiv \mathcal{M}_\text{on}^\text{state} (\text{ref})$', color='#56B4E9')

axs[1].set_ylabel(r'$W_1(X_k \sim P, \hat{X}_k \sim Q)$', fontsize=15)
axs[1].set_xlabel(r'$k$', fontsize=15)
axs[1].tick_params(reset=True, axis='both', which='both', direction='in')
#axs[1].legend(fontsize=12, frameon=False)

fig.tight_layout()
fig.savefig(os.path.join(data_path, 'metrics.pdf'))
plt.show()

### Temporal evolution of the learned neural emulator:

In [ ]:
with h5py.File(os.path.join(data_path, 'datasets.h5'), 'r') as f:
    x_k_ref = np.array(f['x_k_ref'])
    x_k_emu = np.array(f['x_k_emu'])

# "Single-timescale" model, with N_j = 0
eq_single = L96(
    b=eq.b,
    c=eq.c,
    h=eq.h,
    n_k=eq.n_k,
    n_j=0
)

dt_single = dt * eq.n_j

# Baseline "single-timescale" without correction
solver_single = jax.jit(dynamical_solver_single(
    eq_single,
    stepper.RK4(dt_single),
    source_term
))

abstract_model = nnx.eval_shape(lambda: ResMLP(
    in_features=eq.n_k,
    latent=64,
    out_features=eq.n_k,
    n_blocks=3,
    means=jnp.zeros((8,)), 
    stds=jnp.zeros((8,)),
    activation=nnx.relu,
    rngs=nnx.Rngs(42)
))

graph, abstract_state = nnx.split(abstract_model)

checkpoint_path = os.path.join(data_path, 'emu_checkpoint/')
checkpointer = ocp.Checkpointer(ocp.StandardCheckpointHandler())
import warnings
with warnings.catch_warnings(action='ignore'):
    state = checkpointer.restore(checkpoint_path, abstract_state)
eq_emu = nnx.merge(graph, state)

def __explicit__(
    x_k,
    _y_j,
    source
):
    x_dot = eq_emu(x_k) + source    
    return (
        x_dot,
        0
    )

solver_emu = jax.jit(stepper.RK4(dt_single)(
    source_term, 
    __explicit__
))

x_k_nne = np.zeros_like(x_k_ref)
pbar = tqdm.tqdm(range(n_trajs), bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
for traj in pbar:
    x_k_nne[traj, 0] = x_k_ref[traj, 0]
    for step in range(1, n_steps):
        x_k_nne[traj, step], _ = solver_emu(x_k_nne[traj, step - 1], 0, 0)

time_t = np.arange(n_trajs * n_steps) * dt_single

fig, axs = plt.subplots(ncols=1, nrows=1, figsize=(7.5, 3.5), dpi=120)

axs.plot(time_t, np.mean(x_k_ref, axis=-1).flatten(), label=r'$f(X_k, Y_{j,k})$')
axs.plot(time_t, np.mean(x_k_emu, axis=-1).flatten(), '-o', markevery=n_steps, label=r'$g(X_k)$')
axs.plot(time_t, np.mean(x_k_nne, axis=-1).flatten(), linestyle='--', label=r'$\mathcal{E}(X_k)$')
axs.set_xlabel(r'$t$', fontsize=15)
axs.set_ylabel(r'$\langle X_k \rangle_k(t)$', fontsize=15)
axs.tick_params(reset=True, axis='both', which='both', direction='in')
axs.legend(fontsize=12, frameon=False)

fig.tight_layout()
fig.savefig(os.path.join(data_path, 'trajectories.pdf'))
plt.show()

### Experiments; not included in the paper